# Sesion 4 - Agente RAG para Inteligencia de Negocio

En este notebook construiremos un asistente que responde preguntas de negocio usando documentos internos simulados de una empresa ficticia llamada NovaRetail.

El objetivo no es solo ejecutar ChromaDB. El objetivo es entender por que RAG es valioso para agentes empresariales:

1. Un LLM sin contexto puede responder de forma generica o inventar detalles.
2. RAG recupera evidencia desde documentos internos antes de responder.
3. Un agente RAG puede responder con fuentes, limites y recomendaciones mas confiables.

## Preparacion

Antes de ejecutar el notebook, instalar dependencias y descargar los modelos locales:

```bash
poetry install
ollama pull qwen2.5:3b
ollama pull nomic-embed-text
```

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from agents.rag_course_assistant.agent import RAGCourseAssistant
from agents.rag_course_assistant.document_loader import list_markdown_documents, load_markdown_document
from agents.rag_course_assistant.chunker import chunk_markdown_by_sections
from agents.rag_course_assistant.vector_store import ChromaCourseVectorStore, LocalHashEmbeddingFunction

business_docs_dir = project_root / "data" / "rag_business_case"
document_paths = list_markdown_documents(business_docs_dir)
document_paths

[WindowsPath('d:/Documentos/Universidad Externado/Seminario - Multiagentes/data/rag_business_case/definiciones_kpi_comerciales.md'),
 WindowsPath('d:/Documentos/Universidad Externado/Seminario - Multiagentes/data/rag_business_case/politica_descuentos_2026.md'),
 WindowsPath('d:/Documentos/Universidad Externado/Seminario - Multiagentes/data/rag_business_case/reporte_comercial_q3_2026.md')]

## 1. El caso empresarial

NovaRetail tiene documentos internos dispersos: un reporte comercial, una politica de descuentos y un diccionario de KPIs.

Un gerente podria preguntar:

- Por que cayo el margen bruto?
- Que clientes deberia priorizar el equipo comercial?
- Quien aprueba un descuento de 20%?
- Como se define churn mensual?

Un LLM general puede intentar responder, pero no conoce estos documentos. RAG permite que el agente busque primero la evidencia interna.

In [2]:
for path in document_paths:
    print(path.name)
    print(load_markdown_document(path)[:700])
    print("-" * 100)

definiciones_kpi_comerciales.md
# Diccionario de KPIs Comerciales - NovaRetail

## Ventas netas

Ventas netas corresponde al valor de ventas despues de devoluciones, cancelaciones y ajustes comerciales. No incluye impuestos indirectos.

## Margen bruto

Margen bruto se calcula como:

```text
Margen bruto = (Ventas netas - Costo de ventas) / Ventas netas
```

Este indicador mide la rentabilidad antes de gastos operativos, marketing y administracion.

## Churn mensual

Churn mensual corresponde al porcentaje de clientes activos del mes anterior que no realizaron ninguna compra durante el mes actual.

```text
Churn mensual = Clientes perdidos / Clientes activos del mes anterior
```

## Cliente en riesgo

Un cliente se conside
----------------------------------------------------------------------------------------------------
politica_descuentos_2026.md
# Politica Comercial de Descuentos 2026 - NovaRetail

## Objetivo

La politica de descuentos busca proteger la rentabilidad comercial y ev

## 2. Dividir documentos en chunks

Un sistema RAG no envia todos los documentos completos al modelo. Primero divide los documentos en fragmentos recuperables.

Cada chunk conserva metadatos: identificador, fuente, seccion y posicion.

In [3]:
chunks = []
for path in document_paths:
    content = load_markdown_document(path)
    chunks.extend(chunk_markdown_by_sections(content, source=str(path), max_chars=900))

len(chunks), chunks[0]

(18,
 DocumentChunk(chunk_id='chunk_001', text='# Diccionario de KPIs Comerciales - NovaRetail', source='d:\\Documentos\\Universidad Externado\\Seminario - Multiagentes\\data\\rag_business_case\\definiciones_kpi_comerciales.md', section='Diccionario de KPIs Comerciales - NovaRetail', position=1))

In [4]:
for chunk in chunks[:5]:
    print(chunk.chunk_id, "|", chunk.section, "|", len(chunk.text))

chunk_001 | Diccionario de KPIs Comerciales - NovaRetail | 46
chunk_002 | Ventas netas | 155
chunk_003 | Margen bruto | 215
chunk_004 | Churn mensual | 233
chunk_005 | Cliente en riesgo | 310


## 3. Crear embeddings e indexar la base vectorial

Este paso convierte cada chunk en un vector numerico usando embeddings y lo guarda en ChromaDB.

La base vectorial permite responder preguntas por significado, no solo por palabras exactas.

In [5]:
import subprocess

installed_models = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    check=False,
).stdout

if "nomic-embed-text" in installed_models:
    assistant = RAGCourseAssistant(
        model_name="qwen2.5:3b",
        embedding_model_name="nomic-embed-text",
        document_paths=document_paths,
    )
else:
    print("Modelo nomic-embed-text no encontrado. Usando embedding local de respaldo.")
    print("Para usar embeddings semanticos reales, ejecuta: ollama pull nomic-embed-text")
    assistant = RAGCourseAssistant(
        model_name="qwen2.5:3b",
        document_paths=document_paths,
        vector_store=ChromaCourseVectorStore(
            embedding_function=LocalHashEmbeddingFunction(),
        ),
    )


indexed_chunks = assistant.index_course_content(reset=True)
indexed_chunks

18

## 4. Recuperar evidencia para una pregunta de negocio

Antes de pedir una respuesta final al modelo, inspeccionamos que fragmentos fueron recuperados.

Esta es una parte central de RAG: podemos auditar que evidencia recibira el agente.

In [6]:
question = "Por que pudo caer el margen bruto en Q3 y que deberia hacer el equipo comercial?"

retrieved = assistant.vector_store.query(question, top_k=4)

for item in retrieved:
    print(item.chunk_id, "|", item.section, "| distance:", item.distance)
    print(item.text[:500])
    print("-" * 80)

chunk_018 | Riesgos para Q4 | distance: 138.483154296875
## Riesgos para Q4

Los principales riesgos para Q4 son presion sobre margen bruto, aumento del churn en clientes pequenos y deterioro de la experiencia de entrega en picos de demanda.
--------------------------------------------------------------------------------
chunk_014 | Resumen ejecutivo | distance: 181.00906372070312
## Resumen ejecutivo

NovaRetail cerro el tercer trimestre de 2026 con ventas netas por 48.2 millones de dolares, lo que representa un crecimiento de 6.8% frente al trimestre anterior. El crecimiento se concentro en el canal digital y en clientes empresariales medianos.

El margen bruto consolidado fue de 34.6%, inferior al 36.1% observado en Q2. La reduccion se explica principalmente por mayores descuentos comerciales en la categoria de tecnologia y por incremento en costos logisticos de entreg
--------------------------------------------------------------------------------
chunk_009 | Reglas generales | dis

## 5. Comparar respuesta sin RAG y con RAG

La respuesta sin RAG puede sonar razonable, pero no sabe que paso en NovaRetail.

La respuesta con RAG debe basarse en los documentos recuperados y mostrar fuentes.

In [7]:
baseline_answer = assistant.answer_without_rag(question)
print(baseline_answer)

El cese del margen bruto en la Tercera Quincena (Q3) de cualquier empresa puede ser causado por varias razones, incluyendo fluctuaciones en precios, variación en costos de producción, cambios en las políticas comerciales, o problemas con el producto. En general, es importante enfocarse en estas áreas y no solo en los resultados finales del margen bruto.

El equipo comercial podría hacer lo siguiente:

1. Analizar Precios: Verificar si ha habido un cambio significativo en los precios de compra o venta que afecte la rentabilidad.
2. Controlar Costos: Identificar fuentes de aumento inesperado en costos, como subidas salariales, aumentos de factores de producción y gastos operativos.
3. Revisar Precios Actuales: Ver si los precios actuales reflejan adecuadamente el valor del producto o servicio.
4. Investigar Problemas con Productos/Servicios: Identificar problemas en la fabricación o servicio que podrían afectar al margen bruto.
5. Optimización de Ofertas y Precio: Evaluar las estrategias

In [8]:
rag_answer = assistant.answer(question, top_k=4)
print(rag_answer.content)

### Respuesta

Pudo caer el margen bruto en Q3 debido a dos factores principales:

1. Mayor presión sobre los descuentos comerciales, especialmente en la categoría de tecnología.
2. Un aumento de costos de logística para las entregas express.

El equipo comercial debería revisar su política de descuentos y asegurar que estén dentro del rango permitido según las reglas generales establecidas. Adicionalmente, podrían evaluar si hay posibilidades de reducir costos en la categoría de tecnología o mejorar la eficiencia operativa para combatir el aumento en los costos.

### Interpretación

El margen bruto es un indicador crucial que muestra cómo el negocio está reteniendo ganancias antes de gastos de ventas, marketing y administración. El hecho de que ha caído desde Q2 hasta Q3 sugiere que la estrategia comercial o las condiciones del mercado han cambiado.

### Recomendaciones

1. Asegurar que los descuentos comerciales se ajusten al rango permitido (menor a 18%) para mantener un margen brut

## 6. Probar preguntas de negocio

In [ ]:
questions = [
    "Que clientes deberia priorizar el equipo comercial para reducir churn?",
    "Un descuento de 20% puede aprobarse solo por el gerente comercial?",
    "Como se define churn mensual en NovaRetail?",
    "Cual fue el EBITDA de NovaRetail en Q3?",
]

for q in questions:
    print("PREGUNTA:", q)
    response = assistant.answer(q, top_k=4)
    print(response.content)
    print("=" * 100)

PREGUNTA: Que clientes deberia priorizar el equipo comercial para reducir churn?
### Respuesta

Para reducir el churn, NovaRetail debería priorizar a clientes B2B y pequeños clientes con altas frecuencias de compra.

Explicación: Los datos proporcionados destacan dos grupos de clientes con alto riesgo de churn: los clientes pequeños con baja frecuencia de compra y aquellos que han reportado incidencias logísticas. Sin embargo, la segmentación menciona priorizar descuentos en clientes altamente potenciales para recompra, lo cual sugiere un enfoque diferente hacia el cliente B2B.

Además, es importante tener en cuenta la recomendación de reducir el churn mensual promedio a 3.0%, que ha sido superado. Los clientes pequeños y los con incidencias logísticas representan riesgos específicos que deben atenderse.

Fuentes usadas:
- Chunk_016
- Chunk_018

### Recomendaciones

El equipo comercial debería enfocarse en:

1. Atender a clientes B2B, dado su alto potencial para recompra y margen histó

: 

## Discusion final

Preguntas para el grupo:

1. Que diferencia hay entre una recomendacion generica y una recomendacion basada en documentos internos?
2. Que riesgos aparecen si el agente recupera chunks irrelevantes?
3. Como deberia comportarse el agente cuando los documentos no contienen la respuesta?
4. Que documentos reales de una empresa serian buenos candidatos para un primer RAG?
5. En que casos un agente RAG seria mas valioso que un dashboard tradicional?